# Explainable Acne Severity Grading Pipeline (M1–M7) — Unattended Google Colab Runner

This self-contained Jupyter Notebook (`Thesis_Colab.ipynb`) runs the entire **M1–M7 Thesis Pipeline** on Google Colab (recommended runtime: **T4 GPU**) strictly on the empirical Parquet dataset [`Layered-Labs/ACNE04`](https://huggingface.co/datasets/Layered-Labs/ACNE04), with **zero synthetic data or simulated fallbacks**.

### Key Unattended Execution Features
1. **Automatic Per-Runner Isolated Google Drive Folder**:
   - **Step 0** detects if `/content/drive/MyDrive/Acne_Severity_Thesis` is a Google Drive shortcut (`.shortcut-targets-by-id/...`) pointing to another groupmate's folder and automatically creates a new, separate folder in the current runner's own `MyDrive`.
2. **Exclusive `Layered-Labs/ACNE04` Hugging Face Parquet Ingestion**:
   - **Stage 1 (M1)** loads the full [`Layered-Labs/ACNE04`](https://huggingface.co/datasets/Layered-Labs/ACNE04) Parquet dataset (parsing `image`, `level`, `file_name`, and `lesion_boxes`), standardizes all images to **512×512** (aspect-ratio-preserving letterboxing), isolates the **51 watermarked images** into `Classification/JPEGImages_excluded/`, and builds the Stratified 5×2 CV manifest strictly over the training-eligible images.
3. **100% Empirical Pipeline (Zero Legacy Formats or Fallbacks)**:
   - **Stage 2 (M3)**: Runs 19-class BiSeNet (`bisenet_resnet18_celebamaskhq.onnx`) face parsing (`skin=1`, `nose=10`) and `CIELAB` + `CLAHE` preprocessing on every training-eligible 512×512 image.
   - **Stage 3 (M4)**: Extracts the 40-feature matrix (4 whole-ROI `CIELAB a*` erythema stats + 20 luminance `CLAHE L*` masked 32-level GLCM features + 16 scale-normalized, baseline-centered chromatic `CIELAB a*` masked 32-level GLCM features) strictly from the real images and M3 BiSeNet masks.
   - **Stage 4 (M2)**: Trains/evaluates Faster R-CNN ResNet-101 across all 10 5×2 CV folds strictly on the 512×512 images and `Layered-Labs/ACNE04` Parquet lesion bounding boxes (`M1_outputs/m1_lesion_boxes_512.json`).
   - **Stage 5–8 (M5, M6, M7, Single-Image)**: Trains `scikit-learn` `RandomForestClassifier(class_weight="balanced")`, runs isolated-subprocess end-to-end computational benchmarks (true OS Peak RSS, VRAM, model disk footprint, and M3/M4/M5 stage latency), evaluates Quadratic Weighted Kappa ($\kappa_w$ with Fleiss-Cohen-Everitt 95% CI) and the Alpaydın (1999) Combined 5×2 CV F-Test (Macro F1 & QWK), and computes `shap.TreeExplainer` explanations (including full 40-feature importance table and all-features beeswarm plot).

## Step 0: Configure Your Unattended Run, Mount Google Drive & Create Isolated Runner Folder

In [ ]:
#@title 0. Unattended Run Configuration & Isolated Per-Runner Folder Setup { display-mode: "form" }
import os
import sys
import time
import json
import shutil
import logging
from datetime import datetime
from pathlib import Path

# --- Google Drive & Per-Runner Folder Settings ---
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
BASE_PROJECT_FOLDER_NAME = "Acne_Severity_Thesis" #@param {type:"string"}
RUNNER_NAME = "" #@param {type:"string"}
ALWAYS_CREATE_NEW_RUN_FOLDER = False #@param {type:"boolean"}

# --- Hugging Face ACNE04 Parquet Dataset Settings (Layered-Labs/ACNE04 Only) ---
HF_DATASET_ID = "Layered-Labs/ACNE04" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}
LOCAL_PARQUET_PATH = "" #@param {type:"string"}
M1_TARGET_SIZE = 512 #@param {type:"integer"}
M1_RESIZE_MODE = "letterbox" #@param ["letterbox", "stretch"]
EXCLUDE_WATERMARKED_IMAGES = True #@param {type:"boolean"}

# --- Pipeline Execution Settings ---
RUN_M3_PREPROCESSING = True #@param {type:"boolean"}
M3_CHECKPOINT_SYNC_EVERY = 50 #@param {type:"integer"}

RUN_M4_FEATURES = True #@param {type:"boolean"}
TEXTURE_SOURCE = "clahe_l_star" #@param ["clahe_l_star", "grayscale", "lab_l_star"]
BACKUP_ITER1_IF_SCHEMA_CHANGED = True #@param {type:"boolean"}

RUN_M2_BASELINE_RCNN = True #@param {type:"boolean"}
M2_MODE = "train-5x2" #@param ["train-5x2", "infer-pretrained"]
M2_PRETRAINED_ARCH = "faster_rcnn_resnet101" #@param ["faster_rcnn_resnet101", "faster_rcnn_inception_v2"]
M2_EPOCHS_PER_FOLD = 5 #@param {type:"integer"}
M2_BATCH_SIZE = 2 #@param {type:"integer"}
M2_LEARNING_RATE = 0.002 #@param {type:"number"}
M2_SCORE_THRESHOLD = 0.50 #@param {type:"number"}

RUN_M5_RANDOM_FOREST = True #@param {type:"boolean"}
M5_INNER_CV_FOLDS = 3 #@param {type:"integer"}
M5_LOG_EVERY_CANDIDATE = False #@param {type:"boolean"}
M5_GRID_CHECKPOINT_EVERY = 10 #@param {type:"integer"}

RUN_M6_EVALUATION = True #@param {type:"boolean"}
M6_BENCH_WARMUP_RUNS = 3 #@param {type:"integer"}
M6_BENCH_SAMPLES_RF = 30 #@param {type:"integer"}
M6_BENCH_SAMPLES_RCNN_CPU = 10 #@param {type:"integer"}
M6_BENCH_SAMPLES_RCNN_GPU = 30 #@param {type:"integer"}
M6_BENCH_RUN_RCNN_CPU = True #@param {type:"boolean"}
M6_BENCH_RUN_RCNN_GPU = True #@param {type:"boolean"}
RUN_M7_SHAP = True #@param {type:"boolean"}

RUN_SINGLE_IMAGE_DEMO = True #@param {type:"boolean"}
SINGLE_IMAGE_FILENAME = "levle2_93.jpg" #@param {type:"string"}
RESET_ALL_CHECKPOINTS = False #@param {type:"boolean"}

IN_COLAB = "google.colab" in sys.modules

def resolve_isolated_runner_folder(
    base_root: Path,
    base_name: str,
    runner_name: str = "",
    force_new: bool = False,
) -> Path:
    """Create or select a dedicated writable folder in the current runner's own Drive.
    Never follows a Google Drive shortcut (.shortcut-targets-by-id) to another user's folder."""
    clean_base = Path(base_name).name.strip() or "Acne_Severity_Thesis"
    clean_runner = runner_name.strip().replace(" ", "_")

    def _is_shared_shortcut_or_foreign_owner(p: Path) -> bool:
        try:
            if p.is_symlink() or ".shortcut-targets-by-id" in str(p.resolve()):
                return True
            if p.exists():
                for existing_file in list(p.glob("*.py"))[:3]:
                    with open(existing_file, "ab"):
                        pass
                probe = p / f".runner_write_probe_{os.getpid()}"
                probe.write_text("ok", encoding="utf-8")
                probe.unlink()
            return False
        except PermissionError:
            return True
        except Exception:
            return True

    primary = (base_root / f"{clean_base}_{clean_runner}") if clean_runner else (base_root / clean_base)
    if not force_new and not _is_shared_shortcut_or_foreign_owner(primary):
        primary.mkdir(parents=True, exist_ok=True)
        return primary.resolve()

    tag = clean_runner if clean_runner else "Runner"
    for run_num in range(1, 100):
        candidate = base_root / f"{clean_base}_{tag}_{run_num:02d}"
        if candidate.is_symlink():
            continue
        if not candidate.exists():
            candidate.mkdir(parents=True, exist_ok=True)
            print(f"[Isolated Workspace] Created new dedicated folder for current runner: {candidate}")
            return candidate.resolve()
        if not force_new and not _is_shared_shortcut_or_foreign_owner(candidate):
            print(f"[Isolated Workspace] Using runner's own writable folder: {candidate}")
            return candidate.resolve()

    raise RuntimeError("Could not create a writable project folder on Google Drive.")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        print("Mounting Google Drive...")
        drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = resolve_isolated_runner_folder(
        base_root=Path("/content/drive/MyDrive"),
        base_name=BASE_PROJECT_FOLDER_NAME,
        runner_name=RUNNER_NAME,
        force_new=bool(ALWAYS_CREATE_NEW_RUN_FOLDER),
    )
else:
    PROJECT_ROOT = resolve_isolated_runner_folder(
        base_root=Path(".").resolve(),
        base_name=BASE_PROJECT_FOLDER_NAME,
        runner_name=RUNNER_NAME,
        force_new=bool(ALWAYS_CREATE_NEW_RUN_FOLDER),
    )

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_FILE = PROJECT_ROOT / "colab_unattended_run.log"
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)

def log_banner(msg: str) -> None:
    sep = "=" * 78
    logging.info("\n%s\n%s\n%s", sep, msg, sep)

log_banner(f"UNATTENDED PIPELINE INITIALIZED | ISOLATED PROJECT_ROOT = {PROJECT_ROOT}")

## Step 1: Install Required Packages & Verify T4 GPU Runtime

In [ ]:
#@title 1. Install Dependencies & Check Hardware { display-mode: "form" }
import subprocess
import sys

REQUIRED_PACKAGES = [
    "numpy>=1.24.0",
    "pandas>=2.0.0",
    "scipy>=1.10.0",
    "scikit-learn>=1.3.0",
    "scikit-image>=0.21.0",
    "shap>=0.43.0",
    "opencv-python>=4.8.0",
    "Pillow>=10.0.0",
    "onnxruntime>=1.16.0",
    "mediapipe>=0.10.9",
    "matplotlib>=3.7.0",
    "seaborn>=0.12.2",
    "tqdm>=4.66.0",
    "psutil>=5.9.0",
    "datasets>=2.18.0",
    "pyarrow>=14.0.0",
    "huggingface_hub>=0.23.0",
]

if IN_COLAB:
    print("Installing / verifying required packages on Colab...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + REQUIRED_PACKAGES)

import torch
gpu_info = f"CUDA GPU Available: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU Runtime (CUDA not detected)"
logging.info("Environment ready | %s", gpu_info)

## Step 2: Unpack Pipeline Code
Extracts the embedded M1–M7 Python modules into `PROJECT_ROOT` and `LOCAL_CODE_DIR`.

In [ ]:
#@title 2. Unpack Embedded Pipeline Code { display-mode: "form" }
import base64
import io
import zipfile

EMBEDDED_PROJECT_ZIP_B64 = """
...
"""

LOCAL_CODE_DIR = Path("/content/acne_pipeline_code").resolve() if IN_COLAB else (PROJECT_ROOT / "_local_code").resolve()
LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)

def unpack_embedded_project(target_root: Path, local_code_dir: Path) -> None:
    raw_bytes = base64.b64decode(EMBEDDED_PROJECT_ZIP_B64.encode("ascii"))
    with zipfile.ZipFile(io.BytesIO(raw_bytes), "r") as zf:
        for member in zf.infolist():
            parts = Path(member.filename).parts
            if len(parts) <= 1:
                continue
            rel_target = Path(*parts[1:])
            dest_path = target_root / rel_target
            if member.is_dir():
                dest_path.mkdir(parents=True, exist_ok=True)
            else:
                payload = zf.read(member.filename)
                if rel_target.suffix == ".py":
                    (local_code_dir / rel_target.name).write_bytes(payload)
                dest_path.parent.mkdir(parents=True, exist_ok=True)
                if rel_target.suffix == ".py" or not dest_path.exists():
                    dest_path.write_bytes(payload)

unpack_embedded_project(PROJECT_ROOT, LOCAL_CODE_DIR)

for p_str in (str(PROJECT_ROOT), str(LOCAL_CODE_DIR)):
    if p_str in sys.path:
        sys.path.remove(p_str)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(LOCAL_CODE_DIR))
os.chdir(PROJECT_ROOT)

logging.info("Project code modules active in %s (Runner folder: %s):", LOCAL_CODE_DIR, PROJECT_ROOT)
for py_file in sorted(LOCAL_CODE_DIR.glob("*.py")):
    print(f"  - {py_file.name:30s} ({py_file.stat().st_size:,d} bytes)")

## Stage 1 (M1): Load `Layered-Labs/ACNE04` Parquet, Standardize to 512×512 & Exclude Watermarks
Loads the full [`Layered-Labs/ACNE04`](https://huggingface.co/datasets/Layered-Labs/ACNE04) Parquet dataset (parsing `image`, `level`, `file_name`, and `lesion_boxes`), applies 512×512 resolution standardization, isolates the 51 watermarked images into `Classification/JPEGImages_excluded/`, and exports `M1_outputs/m1_dataset_manifest.csv`, `M1_outputs/m1_5x2_fold_assignments.csv`, and `M1_outputs/m1_lesion_boxes_512.json` strictly for the training-eligible dataset.

In [ ]:
#@title Stage 1: Run M1 Data Loader on Layered-Labs/ACNE04 Parquet { display-mode: "form" }
import importlib
import pandas as pd
import m1_data_loader
importlib.reload(m1_data_loader)

log_banner(f"STAGE 1 (M1): Loading Full {HF_DATASET_ID} ({M1_TARGET_SIZE}x{M1_TARGET_SIZE}) & Excluding Watermarks")

m1_result = m1_data_loader.run_m1_pipeline(
    project_root=PROJECT_ROOT,
    dataset_id=HF_DATASET_ID,
    parquet_path=LOCAL_PARQUET_PATH.strip() or None,
    target_size=int(M1_TARGET_SIZE),
    resize_mode=M1_RESIZE_MODE,
    exclude_watermarked=bool(EXCLUDE_WATERMARKED_IMAGES),
    hf_token=HF_TOKEN.strip() or None,
    overwrite=bool(RESET_ALL_CHECKPOINTS),
)

df_full = m1_result["full_manifest"]
df_excluded = m1_result["excluded_manifest"]
df_manifest = m1_data_loader.normalize_manifest_columns(m1_result["dataset_manifest"])
df_folds = m1_data_loader.normalize_manifest_columns(m1_result["fold_assignments"])

grade_col = m1_data_loader.resolve_grade_column(df_manifest)
summary_df = df_manifest.groupby(grade_col).size().reset_index(name="training_image_count")
summary_df["full_dataset_count"] = df_full.groupby(grade_col).size().reindex(summary_df[grade_col]).values
summary_df["excluded_watermarks"] = (
    df_excluded.groupby(grade_col).size().reindex(summary_df[grade_col], fill_value=0).values
)
summary_df["severity_label"] = summary_df[grade_col].map(m1_data_loader.HAYASHI_GRADE_DESCRIPTIONS)
summary_df["training_percentage"] = (summary_df["training_image_count"] / len(df_manifest) * 100).round(2)

logging.info(
    "M1 Verified: %d total loaded | %d watermarked excluded | %d training-eligible | %d 5x2 CV configurations.",
    len(df_full),
    len(df_excluded),
    len(df_manifest),
    df_folds["configuration"].nunique(),
)
display(
    summary_df[
        [
            grade_col,
            "severity_label",
            "full_dataset_count",
            "excluded_watermarks",
            "training_image_count",
            "training_percentage",
        ]
    ]
)

## Stage 2 (M3): BiSeNet 19-Class Facial Skin Segmentation & CIELAB/CLAHE Preprocessing
Downloads `bisenet_resnet18_celebamaskhq.onnx`, runs `512×512` letterbox standardization and 19-class BiSeNet semantic segmentation (`skin=1`, `nose=10`) on every image in `df_manifest`, and saves per-image masks to `M3_outputs/roi_masks/` with resumable checkpointing.

In [ ]:
#@title Stage 2: Run M3 BiSeNet Skin Segmentation & CIELAB Preprocessing { display-mode: "form" }
import urllib.request
import cv2
import numpy as np
from tqdm.auto import tqdm
import m3_common
import m3_bisenet_pipeline
import m3c_face_parsing

if RUN_M3_PREPROCESSING:
    log_banner("STAGE 2 (M3): Running Resumable BiSeNet 19-Class Skin+Nose Segmentation")
    roi_masks_dir = PROJECT_ROOT / "M3_outputs" / "roi_masks"
    masked_imgs_dir = PROJECT_ROOT / "M3_outputs" / "masked_images"
    roi_masks_dir.mkdir(parents=True, exist_ok=True)
    masked_imgs_dir.mkdir(parents=True, exist_ok=True)

    m3_ckpt_file = PROJECT_ROOT / "M3_outputs" / "m3_checkpoint_state.json"
    if RESET_ALL_CHECKPOINTS and m3_ckpt_file.exists():
        m3_ckpt_file.unlink()

    onnx_model_path = PROJECT_ROOT / "models" / "bisenet_resnet18_celebamaskhq.onnx"
    onnx_model_url = "https://github.com/yakhyo/face-parsing/releases/download/weights/resnet18.onnx"
    if not onnx_model_path.is_file():
        onnx_model_path.parent.mkdir(parents=True, exist_ok=True)
        logging.info("Downloading BiSeNet ONNX weights from %s ...", onnx_model_url)
        urllib.request.urlretrieve(onnx_model_url, onnx_model_path)

    bisenet_session = m3c_face_parsing.create_face_parsing_session(onnx_model_path, verify_sha256=False)
    jpg_dir = PROJECT_ROOT / "Classification" / "JPEGImages"

    processed_count = 0
    skipped_count = 0
    for idx, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="M3 BiSeNet Preprocessing"):
        fname = str(row["filename"])
        img_stem = Path(fname).stem
        mask_path = roi_masks_dir / f"{img_stem}.png"
        masked_rgb_path = masked_imgs_dir / f"{img_stem}.png"

        if not RESET_ALL_CHECKPOINTS and mask_path.exists() and masked_rgb_path.exists():
            skipped_count += 1
            continue

        img_path = PROJECT_ROOT / str(row["source_path"])
        if not img_path.is_file():
            img_path = jpg_dir / fname
        if not img_path.is_file():
            raise FileNotFoundError(f"[M3] Image file missing on disk: {img_path}")

        _orig_pil, std_rgb, content_mask, _meta = m3_common.standardize_image(img_path, target_size=512)
        _cls_map, _skin_only, _nose_only, roi_bool = m3_bisenet_pipeline.parse_face(
            std_rgb, content_mask, bisenet_session
        )
        if np.count_nonzero(roi_bool) == 0:
            raise RuntimeError(f"[M3] BiSeNet produced 0 skin+nose pixels for {fname}")

        skin_mask = roi_bool.astype(np.uint8) * 255
        masked_rgb = cv2.bitwise_and(std_rgb, std_rgb, mask=skin_mask)
        cv2.imwrite(str(mask_path), skin_mask)
        cv2.imwrite(str(masked_rgb_path), cv2.cvtColor(masked_rgb, cv2.COLOR_RGB2BGR))
        processed_count += 1

        if processed_count % M3_CHECKPOINT_SYNC_EVERY == 0:
            m3_ckpt_file.write_text(json.dumps({
                "last_filename": fname,
                "processed_new": processed_count,
                "skipped_existing": skipped_count,
                "timestamp": datetime.now().isoformat(),
            }, indent=2))

    logging.info("M3 Preprocessing finished: %d newly processed, %d resumed from checkpoint.", processed_count, skipped_count)
else:
    logging.info("Skipping M3 Preprocessing (RUN_M3_PREPROCESSING=False).")

## Stage 3 (M4): Chromatic (`CIELAB a*`), Luminance GLCM & Scale-Normalized Chromatic `a*` GLCM Feature Extraction
Extracts the **40-feature matrix** (4 whole-ROI `CIELAB a*` erythema stats + 20 masked 32-level `CLAHE L*` GLCM features + 16 tight-cropped `384×384`, baseline-centered masked 32-level `CIELAB a*` chromatic GLCM features) strictly from the physical images and saved M3 BiSeNet ROI masks. If an earlier 24-feature Iteration 1 matrix is detected on Drive, safely backs up `M4_outputs`–`M7_outputs` into `Iteration1_Base24_Backup/` (leaving M1, M2 Faster R-CNN, and M3 ROI masks intact) before extracting the 40-feature matrix.

In [ ]:
#@title Stage 3: Run M4 40-Feature Chromatic & Masked GLCM Extraction { display-mode: "form" }
import importlib
import m4_feature_extraction
importlib.reload(m4_feature_extraction)

feature_csv_path = PROJECT_ROOT / "M4_outputs" / "m4_feature_matrix.csv"
expected_m4_cols = set(m4_feature_extraction.get_feature_columns(distances=(1, 3, 5, 10)))

if RUN_M4_FEATURES:
    log_banner("STAGE 3 (M4): Extracting 40 Chromatic (CIELAB a*) & Masked 32-Level GLCM Features")
    need_extract = True
    if not RESET_ALL_CHECKPOINTS and feature_csv_path.exists():
        df_existing_feat = pd.read_csv(feature_csv_path)
        has_all_cols = expected_m4_cols.issubset(df_existing_feat.columns)
        if len(df_existing_feat) == len(df_manifest) and has_all_cols:
            logging.info(
                "Resumed complete 40-feature M4 matrix from %s (%d rows, %d features).",
                feature_csv_path,
                len(df_existing_feat),
                len(expected_m4_cols),
            )
            need_extract = False
        elif not has_all_cols:
            missing_n = len(expected_m4_cols - set(df_existing_feat.columns))
            logging.warning(
                "Existing %s is missing %d new v6 feature column(s) (e.g., chromatic a* GLCM).",
                feature_csv_path.name,
                missing_n,
            )
            if BACKUP_ITER1_IF_SCHEMA_CHANGED:
                backup_dir = PROJECT_ROOT / "Iteration1_Base24_Backup"
                backup_dir.mkdir(parents=True, exist_ok=True)
                for stage_dir in ("M4_outputs", "M5_outputs", "M6_outputs", "M7_outputs"):
                    src_d = PROJECT_ROOT / stage_dir
                    dst_d = backup_dir / stage_dir
                    if src_d.is_dir() and not dst_d.exists():
                        shutil.move(str(src_d), str(dst_d))
                        logging.info("Backed up Iteration 1 %s -> %s", stage_dir, dst_d)

    if need_extract:
        m4_feature_extraction.run_m4_feature_extraction(
            project_root=PROJECT_ROOT,
            texture_source=TEXTURE_SOURCE,
            distances=(1, 3, 5, 10),
        )

df_features = pd.read_csv(feature_csv_path)
logging.info("M4 Feature Matrix ready: %d images x %d total columns.", df_features.shape[0], df_features.shape[1])
display(df_features.head(5))

## Stage 4 (M2): Baseline Faster R-CNN Lesion-Counting Module ([`wenh06/acne_detection`](https://github.com/wenh06/acne_detection))
Trains and evaluates Faster R-CNN ResNet-101 across all 10 folds of the 5×2 CV split strictly on the training-eligible 512×512 images and `Layered-Labs/ACNE04` Parquet lesion bounding boxes (`M1_outputs/m1_lesion_boxes_512.json`).

In [ ]:
#@title Stage 4: Run M2 Baseline Faster R-CNN (5x2 CV Retraining or Pretrained Inference) { display-mode: "form" }
import importlib
import m2_baseline_rcnn
importlib.reload(m2_baseline_rcnn)

if RUN_M2_BASELINE_RCNN:
    log_banner(f"STAGE 4 (M2): Running Baseline Faster R-CNN | Mode = {M2_MODE} | Arch = {M2_PRETRAINED_ARCH}")
    m2_drive_ckpt = PROJECT_ROOT / "M2_outputs" / "checkpoints"
    m2_drive_ckpt.mkdir(parents=True, exist_ok=True)

    m2_result = m2_baseline_rcnn.train_or_evaluate_m2_5x2(
        project_root=PROJECT_ROOT,
        mode=M2_MODE,
        checkpoint_dir=m2_drive_ckpt,
        output_dir=PROJECT_ROOT / "M2_outputs",
        epochs_per_fold=int(M2_EPOCHS_PER_FOLD),
        batch_size=int(M2_BATCH_SIZE),
        learning_rate=float(M2_LEARNING_RATE),
        score_threshold=float(M2_SCORE_THRESHOLD),
        resume=not bool(RESET_ALL_CHECKPOINTS),
    )

    df_m2_oof = m2_result["predictions_df"]
    m2_state = m2_result["state"]
    logging.info(
        "M2 Completed | Completed Configurations: %d/10 | Total OOF Predictions: %d",
        len(m2_state.get("completed_configurations", [])),
        len(df_m2_oof),
    )
    display(pd.DataFrame(list(m2_state.get("fold_summaries", {}).values())))
else:
    logging.info("Skipping M2 Baseline R-CNN (RUN_M2_BASELINE_RCNN=False).")

## Stage 5 (M5): Balanced Random Forest Classifier with Nested CV Grid Search, Live Logging & Two-Tier Checkpointing
Runs a 192-candidate Nested CV Grid Search (`n_estimators=300`, `max_depth`, `min_samples_leaf`, `max_features`, `class_weight` in `["balanced", "balanced_subsample"]`, `criterion` in `["gini", "entropy"]`) across all 10 folds of the 5×2 CV split with a live `tqdm` progress bar, timestamped Drive logging, and both outer-fold (`m5_training_state.json`) and inner-candidate (`m5_config_XX_grid_checkpoint.json`) checkpointing.

In [ ]:
#@title Stage 5: Train M5 Balanced Random Forest Across All 10 5x2 CV Folds { display-mode: "form" }
import importlib
import m5_random_forest
importlib.reload(m5_random_forest)

if RUN_M5_RANDOM_FOREST:
    log_banner("STAGE 5 (M5): Training Balanced Random Forest Across 10 Folds (Nested CV Grid Search + Live Logging)")
    m5_drive_ckpt = PROJECT_ROOT / "M5_outputs" / "checkpoints"
    m5_drive_ckpt.mkdir(parents=True, exist_ok=True)

    m5_result = m5_random_forest.run_m5_random_forest_pipeline(
        project_root=PROJECT_ROOT,
        feature_matrix_path=PROJECT_ROOT / "M4_outputs" / "m4_feature_matrix.csv",
        output_dir=PROJECT_ROOT / "M5_outputs",
        checkpoint_dir=m5_drive_ckpt,
        inner_cv_folds=int(M5_INNER_CV_FOLDS),
        log_every_candidate=bool(M5_LOG_EVERY_CANDIDATE),
        grid_checkpoint_every=int(M5_GRID_CHECKPOINT_EVERY),
        resume=not bool(RESET_ALL_CHECKPOINTS),
    )

    m5_state = m5_result["state"]
    logging.info(
        "M5 Completed | Best Config: %s (Test Macro-F1 = %.4f) | Saved Best Model: %s",
        m5_state.get("best_configuration"),
        float(m5_state.get("best_macro_f1", 0.0)),
        m5_result["best_model_path"],
    )
    display(pd.DataFrame(list(m5_state.get("completed_configs", {}).values())))
else:
    logging.info("Skipping M5 Random Forest Training (RUN_M5_RANDOM_FOREST=False).")

## Stage 6 (M6): Automated Model Evaluation, Quadratic Weighted Kappa ($\kappa_w$), Hypothesis 1 & 2 Tests, and True End-to-End Speed/Memory Benchmarking
Computes 10-fold Macro F1, Precision, Recall, Per-Class F1, 4×4 Confusion Matrices, OvO Multi-Class AUC-ROC, **Quadratic Weighted Kappa ($\kappa_w$ fold mean $\pm$ SD and pooled $\kappa_w$ with Fleiss-Cohen-Everitt 95% CI & Landis-Koch band)**, **Within ±1 Grade Accuracy**, **Hypothesis 1 Stratified Dummy 5×2 CV F-Test (Macro F1 & QWK)**, **isolated-subprocess End-to-End Computational Benchmarks** (true OS Peak RSS, incremental RAM, GPU VRAM, disk footprint, and M3 BiSeNet / M4 Features / M5 RF stage latency breakdown across RF-Only, Proposed End-to-End CPU, Faster R-CNN CPU, and Faster R-CNN GPU), and the **Alpaydın (1999) Combined 5×2 CV F-Test** (Hypothesis 2 on both Macro F1 and QWK).

In [ ]:
#@title Stage 6: Run M6 Automated Evaluation, QWK Inference, True E2E Benchmarks & 5x2 F-Test { display-mode: "form" }
import importlib
from IPython.display import Image as IPyImage, display
import m6_model_evaluation
importlib.reload(m6_model_evaluation)

if RUN_M6_EVALUATION:
    log_banner("STAGE 6 (M6): Running 10-Fold Evaluation, QWK Inference, Isolated E2E Benchmarks & 5x2 CV F-Tests")
    m6_report = m6_model_evaluation.run_m6_evaluation(
        project_root=PROJECT_ROOT,
        rf_predictions_path=PROJECT_ROOT / "M5_outputs" / "m5_rf_5x2_oof_predictions.csv",
        rcnn_predictions_path=PROJECT_ROOT / "M2_outputs" / "m2_rcnn_5x2_oof_predictions.csv",
        output_dir=PROJECT_ROOT / "M6_outputs",
        warmup_runs=int(M6_BENCH_WARMUP_RUNS),
        n_samples_rf=int(M6_BENCH_SAMPLES_RF),
        n_samples_rcnn_cpu=int(M6_BENCH_SAMPLES_RCNN_CPU),
        n_samples_rcnn_gpu=int(M6_BENCH_SAMPLES_RCNN_GPU),
        run_rcnn_cpu=bool(M6_BENCH_RUN_RCNN_CPU),
        run_rcnn_gpu=bool(M6_BENCH_RUN_RCNN_GPU),
        score_threshold=float(M2_SCORE_THRESHOLD),
    )

    qwk_csv = Path(m6_report["qwk_summary_csv"])
    if qwk_csv.exists():
        print("--- M6 Quadratic Weighted Kappa (QWK) & Ordinal Agreement Summary ---")
        display(pd.read_csv(qwk_csv))

    benchmark_csv = Path(m6_report["true_end_to_end_benchmark_csv"])
    if benchmark_csv.exists():
        print("--- M6 True End-to-End Computational Benchmark (Isolated OS Peak RSS, VRAM & Stage Latency) ---")
        bm_df = pd.read_csv(benchmark_csv)
        bm_cols = [
            c for c in [
                "pipeline", "device", "num_samples", "disk_size_mb",
                "baseline_rss_mb", "peak_rss_mb", "incremental_peak_ram_mb",
                "peak_gpu_vram_mb", "total_working_memory_mb",
                "latency_mean_ms", "latency_std_ms", "latency_p50_ms", "latency_p95_ms",
                "throughput_img_per_sec", "m3_bisenet_ms", "m4_features_ms", "m5_rf_ms",
            ]
            if c in bm_df.columns
        ]
        display(bm_df[bm_cols])

    print("--- M6 Hypothesis 1 & Hypothesis 2 Statistical Tests (Macro F1 & QWK) ---")
    h1 = m6_report.get("hypothesis_1_clinical_efficacy", {})
    h2_f1 = m6_report.get("combined_5x2_f_test_macro_f1", {})
    h2_qwk = m6_report.get("combined_5x2_f_test_qwk", {})
    h1_qwk = h1.get("rf_qwk_pooled_inference", {})
    rc_qwk = h1.get("rcnn_qwk_pooled_inference", {})
    display(pd.DataFrame([
        {
            "test": "H1: Proposed RF vs. Stratified Dummy (Macro F1)",
            "rf_score": f"{h1.get('rf_macro_f1_mean', 0):.4f} ± {h1.get('rf_macro_f1_std', 0):.4f}",
            "comparator_score": f"{h1.get('stratified_dummy_macro_f1_mean', 0):.4f}",
            "F_10_5": h1.get("rf_vs_dummy_5x2_f_stat"),
            "p_value": f"{h1.get('rf_vs_dummy_5x2_p_value', 1.0):.4g}",
            "interpretation": f"Beats chance (alpha=0.05): {h1.get('beats_chance_alpha_005')}",
        },
        {
            "test": "H1: Proposed RF vs. Stratified Dummy (QWK)",
            "rf_score": f"{h1_qwk.get('qwk_pooled', 0):.4f} [{h1_qwk.get('qwk_ci95_low', 0):.4f}, {h1_qwk.get('qwk_ci95_high', 0):.4f}]",
            "comparator_score": f"{m6_report.get('stratified_dummy_summary', {}).get('pooled_qwk', 0):.4f}",
            "F_10_5": h1.get("rf_vs_dummy_qwk_5x2_f_stat"),
            "p_value": f"{h1.get('rf_vs_dummy_qwk_5x2_p_value', 1.0):.4g}",
            "interpretation": f"{h1_qwk.get('landis_koch_band', '')} | Within ±1: {float(h1.get('rf_within_1_grade_acc_mean', 0)):.2%}",
        },
        {
            "test": "H2: Proposed RF vs. Baseline Faster R-CNN (Macro F1)",
            "rf_score": f"{h1.get('rf_macro_f1_mean', 0):.4f} ± {h1.get('rf_macro_f1_std', 0):.4f}",
            "comparator_score": f"{m6_report['faster_rcnn_summary']['macro_f1_mean']:.4f} ± {m6_report['faster_rcnn_summary']['macro_f1_std']:.4f}",
            "F_10_5": h2_f1.get("f_statistic"),
            "p_value": f"{h2_f1.get('p_value', 1.0):.4g}",
            "interpretation": h2_f1.get("model_adoption_decision", ""),
        },
        {
            "test": "H2: Proposed RF vs. Baseline Faster R-CNN (QWK)",
            "rf_score": f"{h1_qwk.get('qwk_pooled', 0):.4f} ({h1.get('rf_qwk_fold_mean', 0):.4f} ± {h1.get('rf_qwk_fold_std', 0):.4f})",
            "comparator_score": f"{rc_qwk.get('qwk_pooled', 0):.4f} ({h1.get('rcnn_qwk_fold_mean', 0):.4f} ± {h1.get('rcnn_qwk_fold_std', 0):.4f})",
            "F_10_5": h2_qwk.get("f_statistic"),
            "p_value": f"{h2_qwk.get('p_value', 1.0):.4g}",
            "interpretation": h2_qwk.get("model_adoption_decision", ""),
        },
    ]))

    summary_csv = PROJECT_ROOT / "M6_outputs" / "m6_model_evaluation_summary.csv"
    if summary_csv.exists():
        print("--- M6 10-Fold Classification, QWK & True E2E Efficiency Summary ---")
        display(pd.read_csv(summary_csv))

    eval_plot_path = Path(m6_report["comparison_figure"])
    if eval_plot_path.exists():
        display(IPyImage(filename=str(eval_plot_path), width=1020))

    bm_plot_path = Path(m6_report["benchmark_figure"])
    if bm_plot_path.exists():
        display(IPyImage(filename=str(bm_plot_path), width=1050))
else:
    logging.info("Skipping M6 Model Evaluation (RUN_M6_EVALUATION=False).")

## Stage 7 (M7): SHAP `TreeExplainer` Global & Local Interpretability
Computes exact TreeSHAP values via `shap.TreeExplainer` across **all 40 features** on the best Random Forest model and generates:
- Compact Top-12 Global plots (`m7_global_shap_bar.png`, `m7_global_shap_beeswarm.png`)
- Separate **All-Features** outputs: full feature importance table (`m7_all_features_importance_table.csv` + `m7_all_features_importance_table.png`) and full all-features directional beeswarm plot (`m7_global_shap_beeswarm_all_features.png`)
- Per-grade Local Waterfall & Force plots (`local_explanations/`)

In [ ]:
#@title Stage 7: Run M7 SHAP TreeExplainer & Display Global / All-Features / Local Plots { display-mode: "form" }
import importlib
from IPython.display import Image as IPyImage, display
import m7_shap_interpretability
importlib.reload(m7_shap_interpretability)

if RUN_M7_SHAP:
    log_banner("STAGE 7 (M7): Computing SHAP TreeExplainer Global & Local Explanations")
    m7_report = m7_shap_interpretability.run_m7_shap_analysis(
        project_root=PROJECT_ROOT,
        best_model_path=PROJECT_ROOT / "M5_outputs" / "models" / "rf_best_model.pkl",
        feature_matrix_path=PROJECT_ROOT / "M4_outputs" / "m4_feature_matrix.csv",
        output_dir=PROJECT_ROOT / "M7_outputs",
    )

    logging.info(
        "M7 Verified: TreeSHAP computed across all %d features | Outputs saved to %s",
        m7_report["n_features"],
        PROJECT_ROOT / "M7_outputs",
    )

    print(f"--- M7 Complete SHAP Feature Importance Table (All {m7_report['n_features']} Features) ---")
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 160):
        display(m7_report["importance_df"])

    bar_plot = Path(m7_report["global_bar_plot"])
    bee_plot = Path(m7_report["global_beeswarm_plot"])
    all_table_img = Path(m7_report["all_features_table_image"])
    all_bee_plot = Path(m7_report["global_beeswarm_all_features_plot"])

    if bar_plot.exists():
        print("--- Compact Global SHAP Bar Plot (Top 12) ---")
        display(IPyImage(filename=str(bar_plot), width=900))
    if bee_plot.exists():
        print("--- Compact Global SHAP Beeswarm Plot (Top 12) ---")
        display(IPyImage(filename=str(bee_plot), width=900))
    if all_table_img.exists():
        print(f"--- Separate Output: All-Features Importance Table ({all_table_img.name}) ---")
        display(IPyImage(filename=str(all_table_img), width=1050))
    if all_bee_plot.exists():
        print(f"--- Separate Output: All-Features Beeswarm Plot ({all_bee_plot.name}) ---")
        display(IPyImage(filename=str(all_bee_plot), width=950))
else:
    logging.info("Skipping M7 SHAP Interpretability (RUN_M7_SHAP=False).")

## Stage 8: Single-Image End-to-End Pipeline & Dual-Model Evaluator
Evaluates a real image from `Classification/JPEGImages/` through `512×512` standardization, BiSeNet ROI masking, `CIELAB` channel visualization, Random Forest vs. Faster R-CNN grading, and local SHAP Waterfall/Force plots.

In [ ]:
#@title Stage 8: Evaluate Single Real Image End-to-End (Preprocessing + CIELAB + RF vs. R-CNN + SHAP) { display-mode: "form" }
import importlib
from IPython.display import Image as IPyImage, display
import evaluate_single_image
importlib.reload(evaluate_single_image)

if RUN_SINGLE_IMAGE_DEMO:
    target_fname = SINGLE_IMAGE_FILENAME if SINGLE_IMAGE_FILENAME.lower().endswith((".jpg", ".jpeg", ".png")) else f"{SINGLE_IMAGE_FILENAME}.jpg"
    candidate_p = PROJECT_ROOT / "Classification" / "JPEGImages" / target_fname
    if not candidate_p.is_file():
        target_fname = str(df_manifest["filename"].iloc[0])
    log_banner(f"STAGE 8: Running Single-Image End-to-End Evaluation for '{target_fname}'")
    single_report = evaluate_single_image.evaluate_single_image_pipeline(
        project_root=PROJECT_ROOT,
        filename=target_fname,
        output_dir=PROJECT_ROOT / "Single_Image_Evaluation",
        score_threshold=float(M2_SCORE_THRESHOLD),
    )

    vis_files = single_report.get("generated_visual_files", {})
    for key in ["preprocessing_cielab_sheet", "evaluation_dashboard", "local_shap_waterfall", "local_shap_force"]:
        if key in vis_files:
            plot_path = Path(vis_files[key])
            if plot_path.exists():
                print(f"--- {plot_path.name} ---")
                display(IPyImage(filename=str(plot_path), width=980))
else:
    logging.info("Skipping Single-Image Demo (RUN_SINGLE_IMAGE_DEMO=False).")

## Stage 9: Bundle All Generated Results, Plots & Trained Models for Download

In [ ]:
#@title Stage 9: Create Final Results Archive on Google Drive { display-mode: "form" }
bundle_zip_path = PROJECT_ROOT / "Acne_Pipeline_Results_Bundle.zip"
output_folders = [
    "M1_outputs",
    "M2_outputs",
    "M4_outputs",
    "M5_outputs",
    "M6_outputs",
    "M7_outputs",
    "Single_Image_Evaluation",
]

with zipfile.ZipFile(bundle_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    if LOG_FILE.exists():
        zf.write(LOG_FILE, LOG_FILE.name)
    for folder in output_folders:
        folder_path = PROJECT_ROOT / folder
        if not folder_path.exists():
            continue
        for root, _, files in os.walk(folder_path):
            for f in sorted(files):
                full_p = Path(root) / f
                rel_p = full_p.relative_to(PROJECT_ROOT)
                zf.write(full_p, rel_p)

log_banner(
    f"ALL PIPELINE STAGES COMPLETED SUCCESSFULLY!\n"
    f"Results Bundle Saved: {bundle_zip_path} ({bundle_zip_path.stat().st_size / (1024*1024):.2f} MB)"
)